# Demo: GNN music-context inference

Load one cached test graph, run the trained audio-only GNN, and compare
predicted tags with the ground truth. The description is displayed as metadata;
this notebook does not run BERT fusion.

Requires the Task 3 graph cache, label mapping, checkpoint, and thresholds.


In [10]:
import json, sys
from pathlib import Path

import numpy as np
import torch

ROOT = Path("..").resolve()          # notebook lives in notebooks/
sys.path.insert(0, str(ROOT / "src"))

GRAPHS  = ROOT / "data/processed/mtat/graphs"
LABELS  = ROOT / "data/processed/mtat/label_space.json"
RESULTS = ROOT / "results/task3_gnn_only/mfcc_tau"

print("src    :", (ROOT / "src").exists())
print("graphs :", GRAPHS.exists())
print("labels :", LABELS.exists())
print("results:", RESULTS.exists())

src    : True
graphs : True
labels : True
results: True


## 1. Load the label space and one test graph

In [11]:
space   = json.loads(LABELS.read_text())
labels  = space["genre"] + space["mood"]
n_genre = space["n_genre"]
print(f"{n_genre} genre + {space['n_mood']} mood = {len(labels)} targets")

test = torch.load(GRAPHS / "test_mfcc_tau.pt", weights_only=False)
g = test[0]

print(f"\nclip {g.track_id}")
print(f"  x          {tuple(g.x.shape)}   ({g.num_segments} segments)")
print(f"  edge_index {tuple(g.edge_index.shape)}")
print(f"  text       {g.text}")
print(f"  true tags  {[t for t, v in zip(labels, g.y[0].tolist()) if v]}")

36 genre + 17 mood = 53 targets

clip 29
  x          (5, 26)   (5 segments)
  edge_index (2, 10)
  text       A music clip featuring violin.
  true tags  ['classical']


## 2. Inspect the graph

Temporal edges chain consecutive segments; similarity edges connect
non-adjacent segments whose track-centred cosine similarity exceeded the
threshold. Those are the structural repetitions the GNN is meant to exploit.

In [12]:
ei, ea = g.edge_index, g.edge_attr
seen = set()
print("temporal edges:")
for k in range(ei.shape[1]):
    a, b = int(ei[0, k]), int(ei[1, k])
    if (b, a) in seen: continue
    seen.add((a, b))
    is_temporal, sim = ea[k].tolist()
    if is_temporal > 0.5:
        print(f"  segment {a} <-> {b}")
print("similarity edges:")
found = False
for k in range(ei.shape[1]):
    a, b = int(ei[0, k]), int(ei[1, k])
    is_temporal, sim = ea[k].tolist()
    if is_temporal <= 0.5 and a < b:
        print(f"  segment {a} <-> {b}   similarity {sim:.3f}")
        found = True
if not found:
    print("  none -- this clip has a bare temporal chain (35% of clips at tau=0.3)")

temporal edges:
  segment 0 <-> 1
  segment 1 <-> 2
  segment 2 <-> 3
  segment 3 <-> 4
similarity edges:
  segment 2 <-> 4   similarity 0.662


## 3. Load the trained model and predict

In [13]:
from gnn import GraphSAGE, DEVICE, HIDDEN

model = GraphSAGE(g.x.shape[1], HIDDEN, len(labels)).to(DEVICE)
model.load_state_dict(torch.load(RESULTS / "best_gnn.pt", map_location=DEVICE))
model.eval()

# per-tag thresholds, tuned on validation during training and applied frozen
thresholds = np.load(RESULTS / "thresholds.npy")

from torch_geometric.loader import DataLoader
batch = next(iter(DataLoader([g], batch_size=1))).to(DEVICE)
with torch.no_grad():
    probs = torch.sigmoid(model(batch)).cpu().numpy()[0]

pred = probs >= thresholds
print(f"predicted: {[t for t, v in zip(labels, pred) if v]}")
print(f"true     : {[t for t, v in zip(labels, g.y[0].tolist()) if v]}")

predicted: ['classical', 'medieval', 'slow']
true     : ['classical']


## 4. Top predictions with confidence

In [14]:
order = np.argsort(-probs)[:10]
print(f"{'tag':<22}{'prob':>8}{'thresh':>8}{'fired':>7}{'true':>6}")
print("-" * 51)
truth = g.y[0].tolist()
for i in order:
    print(f"{labels[i]:<22}{probs[i]:>8.3f}{thresholds[i]:>8.2f}"
          f"{str(bool(probs[i] >= thresholds[i])):>7}{str(bool(truth[i])):>6}")

tag                       prob  thresh  fired  true
---------------------------------------------------
classical                0.841    0.72   True  True
slow                     0.623    0.60   True False
soft                     0.578    0.60  False False
ambient                  0.490    0.74  False False
quiet                    0.465    0.66  False False
fast                     0.327    0.61  False False
opera                    0.298    0.93  False False
new age                  0.272    0.52  False False
baroque                  0.242    0.52  False False
calm                     0.224    0.33  False False


## 5. Saved model comparison

The table uses the retained per-run test metrics and validation-tuned thresholds.

| Model | AUC-PR | Genre F1 | Mood F1 |
|---|---|---|---|
| Prior baseline | 0.0392 | 0.0716 | 0.0683 |
| MLP, no graph | 0.2464 | 0.2843 | 0.2101 |
| GNN, tau = 0.3 | 0.2350 | 0.2965 | 0.1890 |
| GNN, top-k = 2 | 0.2368 | 0.2928 | 0.1861 |

These saved GNN runs have higher genre F1 and lower mood F1 than the MLP,
while overall AUC-PR is lower. The table does not establish significance
across repeated training runs.
